## Team members and responsibilities

| Member | Name | Student ID | Responsibilities |
|---|---|---|---|
| 1 | | | |
| 2 |Tang Shaoyan|A0354555X|1.3,1.4|
| 3 | | | |
| 4 | | | |

## Question 1 — Data Cleaning & Exploring the Data (10 marks)

### 1.1 — Lot-type meanings

**Question from the project instructions**

Look at the features in the dataset. What do the values in the column “lot_type” mean?

The `lot_type` column identifies the vehicle category of the parking lots:

- **C**: Cars.
- **Y**: Motorcycles.
- **H**: Heavy vehicles.
- **S**: Motorcycles with sidecar.

Different entries within one carpark report availability separately by lot type.
These definitions come from the [Carpark Availability API specification](https://data.gov.sg/datasets/d_ca933a644e55d34fe21f28b8052fac63/view),
rather than the supplementary carpark-metadata table.

### 1.2 — Approximate the data-update frequency

**Question from the project instructions**


Carry out and document a systematic approach to approximate the frequency at which
the data values are updated. The purpose is to avoid querying for data unnecessarily.
Although the API date_time parameter is specified to seconds, the database may not
be updated every second.

**Method**

1. Select carpark **HE12** and a short historical window beginning at **12 July 2024, 12:00 SGT**.
2. Make **15 queries at one-minute historical intervals** (12:00–12:14).
3. Record the carpark-level `update_datetime` returned by each query.
4. Remove repeated update timestamps and calculate consecutive differences in seconds.

**Conclusion**

For this carpark and window, the observed distinct update intervals range from
**58 to 150 seconds**, with a mean of approximately
**81.8 seconds**. Updates are irregular in this sample, rather than advancing
once per second.

A **one-minute querying interval** is a reasonable starting point for this project,
consistent with the [official API recommendation](https://data.gov.sg/datasets/d_ca933a644e55d34fe21f28b8052fac63/view).
Two-minute sampling reduces requests but loses temporal resolution. This short,
single-carpark study estimates observed refresh spacing, not a universal backend
frequency: one-minute queries can miss faster updates, and a refreshed timestamp
does not necessarily mean the availability count changed.

In [1]:
import time
import requests
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import display

# Historical query times and update_datetime are interpreted in Singapore time.
start_time = datetime(2024, 7, 12, 12, 0, 0)
carpark_id = "HE12"
results = []
endpoint = "https://api.data.gov.sg/v1/transport/carpark-availability"

print(f"Tracking updates for carpark {carpark_id} using 15 one-minute historical queries...")
with requests.Session() as session:
    for i in range(15):
        if i:
            time.sleep(2)  # Pace HTTP requests independently of historical query spacing.
        current_time = start_time + timedelta(minutes=i)
        time_str = current_time.strftime("%Y-%m-%dT%H:%M:%S")
        response = session.get(endpoint, params={"date_time": time_str}, timeout=30)
        response.raise_for_status()
        data = response.json()
        if not data.get("items"):
            raise ValueError(f"No API snapshot returned for {time_str}.")
        snapshot = data["items"][0]
        carpark = next((cp for cp in snapshot["carpark_data"]
                        if cp["carpark_number"] == carpark_id), None)
        if carpark is None:
            raise ValueError(f"Carpark {carpark_id} is missing at {time_str}.")
        # update_datetime belongs to the carpark object, not carpark_info[0].
        update_dt = carpark["update_datetime"]
        results.append({
            "query_time_SGT": time_str,
            "returned_snapshot": snapshot["timestamp"],
            "update_datetime": update_dt,
        })

df_updates = pd.DataFrame(results)
df_updates["update_datetime"] = pd.to_datetime(df_updates["update_datetime"], errors="raise")
if not df_updates["update_datetime"].is_monotonic_increasing:
    raise ValueError("Update timestamps moved backwards; inspect the responses before inferring a frequency.")
unique_updates = df_updates["update_datetime"].drop_duplicates().reset_index(drop=True)
time_diffs = unique_updates.diff().dt.total_seconds().dropna()

display(df_updates)
print("Consecutive distinct update-time differences (seconds):", time_diffs.tolist())
if time_diffs.empty:
    raise ValueError("No update change was observed; extend the historical sampling window.")
print("Minimum observed interval (seconds):", float(time_diffs.min()))
print("Maximum observed interval (seconds):", float(time_diffs.max()))
print("Mean observed interval (seconds):", float(time_diffs.mean()))

update_summary = {
    "carpark": carpark_id,
    "sample_count": len(df_updates),
    "distinct_updates": len(unique_updates),
    "start": df_updates["query_time_SGT"].iloc[0],
    "end": df_updates["query_time_SGT"].iloc[-1],
    "minimum_seconds": float(time_diffs.min()),
    "maximum_seconds": float(time_diffs.max()),
    "mean_seconds": float(time_diffs.mean()),
}

Tracking updates for carpark HE12 using 15 one-minute historical queries...


,query_time_SGT,returned_snapshot,update_datetime
0,2024-07-12T12:00:00,2024-07-12T11:58:27+08:00,2024-07-12 11:58:13
1,2024-07-12T12:01:00,2024-07-12T12:00:27+08:00,2024-07-12 11:59:13
2,2024-07-12T12:02:00,2024-07-12T12:01:26+08:00,2024-07-12 12:00:13
3,2024-07-12T12:03:00,2024-07-12T12:02:27+08:00,2024-07-12 12:01:45
4,2024-07-12T12:04:00,2024-07-12T12:03:27+08:00,2024-07-12 12:02:43
5,2024-07-12T12:05:00,2024-07-12T12:03:27+08:00,2024-07-12 12:02:43
6,2024-07-12T12:06:00,2024-07-12T12:05:27+08:00,2024-07-12 12:04:43
7,2024-07-12T12:07:00,2024-07-12T12:06:27+08:00,2024-07-12 12:06:13
8,2024-07-12T12:08:00,2024-07-12T12:07:27+08:00,2024-07-12 12:07:13
9,2024-07-12T12:09:00,2024-07-12T12:07:27+08:00,2024-07-12 12:07:13


Consecutive distinct update-time differences (seconds): [60.0, 60.0, 92.0, 58.0, 120.0, 90.0, 60.0, 60.0, 150.0, 60.0, 90.0]
Minimum observed interval (seconds): 58.0
Maximum observed interval (seconds): 150.0
Mean observed interval (seconds): 81.81818181818181


### 1.3 — Unique carpark coverage and changes over time

**Question from the project instructions**


(i) How many unique carparks are included in the carpark availability dataset?
(ii) Check if this value varies with time. Explain why this check is important
(about 20 words).

Counts (12:00 SGT queries): 1 July 2026—1995; 15 July—1997; 31 July—1998. Coverage varies.

Checking changing carpark coverage prevents misleading comparisons caused by missing, added, or removed carparks rather than genuine changes in availability.

### 1.4 — A suspected sensor malfunction

**Question from the project instructions**


A carpark may have malfunctioning sensors. There are many types of possible
malfunctions. Identify one carpark that you believe has malfunctioning sensors.
Explain what the “malfunction” is in this case (about 20 words).

Carpark A88 repeatedly reports 64 available Y-type lots against only 46 total, suggesting erroneous sensor counting or faulty data reporting.

### 1.5 — Hourly July 2026 availability and overall pattern

**Question from the project instructions**


Create a dataset of hourly carpark availability (`lots_available / total_lots`)
from the raw data for **July 2026**. Plot the average (across ALL carparks) hourly
availability against time and identify patterns in about **50 words**.

Decide and document what to do when an hour has no readings (for example, impute
or ignore). Also decide whether to (i) average the per-carpark ratios or (ii)
divide total available lots by total capacity across carparks.

**Team answer / discussion**

**Team implementation / outputs**

### 1.6 — Find the most-correlated carpark

**Question from the project instructions**


Intuitively, availability across certain carparks may be correlated; for example,
many housing carparks may have higher availability during working hours. Using the
same July 2026 interval as Q1.5, pick one carpark and find the carpark most
correlated with it in terms of availability. State the correlation type used
(for example, Pearson or Spearman).

**Team answer / discussion**

**Team implementation / outputs**

### 1.7 — Group proposal for Question 3

**Question from the project instructions**


Include a proposal of around **500 words** explaining what the team intends to do
for the group-proposed project in Question 3. Additional datasets may supplement
the analysis. Present the proposal using markdown in this notebook.

**Team answer / discussion**

## Question 2 — Forecasting (10 marks)

### 2.1 — Motivation and beneficiaries

**Question from the project instructions**


Explain why forecasting future hourly carpark availability is useful, who would
find the information valuable, and what could be done with a good forecasting model.

**Team answer / discussion**

### 2.2 — Ridge regression for PL57

**Question from the project instructions**


Build a ridge-regression model to forecast hourly availability for **PL57**.
Use **July 2026 for training** and **August 2026 for testing**. Do not use
additional datasets in this part; decide which features to use.

Generate (i) a time-series plot of actual and predicted hourly values and
(ii) a scatter plot of actual versus predicted values, including a line showing
how good the fit is.

**Team answer / discussion**

**Team implementation / outputs**

### 2.3 — Random forest for PL57

**Question from the project instructions**


Repeat Q2.2 using a random-forest regressor. Clearly state **all parameters** of
the chosen model. Retain the same training/testing months and required plots.

**Team answer / discussion**

**Team implementation / outputs**

### 2.4 — Multi-layer perceptron for PL57

**Question from the project instructions**


Repeat Q2.2 using a multi-layer perceptron (MLP). Clearly state **all parameters**
of the chosen model. Retain the same training/testing months and required plots.

**Team answer / discussion**

**Team implementation / outputs**

### 2.5 — Tune, compare and recommend one model

**Question from the project instructions**


Choose suitable performance metrics and carry out a simple grid search for
**all three methods**, rather than comparing their default settings alone.
Recommend exactly **one** model using both quantitative and qualitative arguments.

**Team answer / discussion**

**Team implementation / outputs**

### 2.6 — Generate 2928 future hourly predictions

**Question from the project instructions**


Use the selected model to generate a Python-format list of **2928 predictions**:
`24 * (30 + 31 + 30 + 31)`, predicting PL57 hourly availability from
**1 September 2026 through 31 December 2026** (for example, `[0.82, 0.81, ...]`).

**Team answer / discussion**

**Team implementation / outputs**

## Question 3 — Group Proposed Project (10 marks)

### 3.1 — Implement the group project

**Question from the project instructions**


Implement the team's proposed project and explore some aspects of machine-learning
models. You must use the provided dataset, but may supplement it with other data
(for example, weather or free-parking information). The project may expand beyond
the original Q1.7 proposal.

**Team answer / discussion**

**Team implementation / outputs**

### 3.2 — A practical action benefiting society

**Question from the project instructions**


Based on the analysis, suggest a practical action in approximately **200 words**.
The action should benefit society; do not propose technical steps such as
hyperparameter tuning as the action. Implementation of the practical action is
not required. For example, a public forecasting application could help reduce
carpark congestion during peak hours.

**Team answer / discussion**

## Question 4 — Slides and Viva (10 marks)

### 4.1 — Five slides about Question 3

**Question from the project instructions**


Prepare **five slides** covering the team's contribution to **Question 3 only**.
Do **not** cover Questions 1 and 2 in the slides.

**Team answer / discussion**

### 4.2 — Required presentation coverage

**Question from the project instructions**


Include these points at a minimum, with equal emphasis:

1. **Problem Description:** the problem, context and why it matters.
2. **Experimental Setup:** assumptions and experiment setup.
3. **Results and Analysis:** clear visualisations, insights and key reflections.
4. **Reflections and Conclusions:** practical benefits, societal value, takeaway
   message and lessons learned.
5. Present the slides **live in class** and be prepared to defend the work.

**Team answer / discussion**